# Silver - type enforcement, dedup, anomaly rules

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_typed AS
SELECT
  CAST(session_id AS STRING)        AS session_id,
  CAST(station_id AS STRING)        AS station_id,
  CAST(started_at AS TIMESTAMP)     AS started_at,
  CAST(ended_at AS TIMESTAMP)       AS ended_at,
  CAST(duration_minutes AS DOUBLE)  AS duration_minutes,
  CAST(energy_kwh AS DOUBLE)        AS energy_kwh,
  CAST(station_power_kw AS DOUBLE)  AS station_power_kw,
  session_type,
  CAST(price_per_kwh AS DOUBLE)     AS price_per_kwh,
  CAST(cost_pln AS DOUBLE)          AS cost_pln,
  driver_id,
  ingested_at
FROM workspace.default.bronze_sessions;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_dedup AS
SELECT * EXCEPT (rn) FROM (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY session_id ORDER BY ingested_at DESC) AS rn
  FROM workspace.default.silver_typed
) WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.silver_clean AS
SELECT
  *,
  CASE
    WHEN energy_kwh IS NULL OR duration_minutes IS NULL OR price_per_kwh IS NULL
                                                                  THEN 'null_critical_field'
    WHEN ended_at < started_at                                    THEN 'reversed_timestamp'
    WHEN energy_kwh < 0 OR duration_minutes < 0                   THEN 'negative_value'
    WHEN energy_kwh = 0 OR duration_minutes = 0                   THEN 'zero_session_aborted'
    WHEN duration_minutes > 0
         AND energy_kwh > station_power_kw * (duration_minutes/60.0) * 0.95
                                                                  THEN 'exceeds_physical_limit'
    ELSE 'ok'
  END AS anomaly_reason,
  CASE
    WHEN energy_kwh IS NULL OR duration_minutes IS NULL OR price_per_kwh IS NULL THEN TRUE
    WHEN ended_at < started_at                                                   THEN TRUE
    WHEN energy_kwh < 0 OR duration_minutes < 0                                  THEN TRUE
    WHEN energy_kwh = 0 OR duration_minutes = 0                                  THEN TRUE
    WHEN duration_minutes > 0
         AND energy_kwh > station_power_kw * (duration_minutes/60.0) * 0.95      THEN TRUE
    ELSE FALSE
  END AS is_anomaly
FROM workspace.default.silver_typed
QUALIFY ROW_NUMBER() OVER (PARTITION BY session_id ORDER BY ingested_at DESC) = 1;

In [0]:
%sql
SELECT anomaly_reason, COUNT(*) AS cnt,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM workspace.default.silver_clean
GROUP BY anomaly_reason ORDER BY cnt DESC;